In [ ]:
!pip install --upgrade pip setuptools
!pip install nltk sacrebleu pandas pyter
import nltk
nltk.download('punkt')

  Using cached pyter-0.2.2.1.tar.gz (6.6 kB)
  Preparing metadata (setup.py) ... done
  Using cached distribute-0.7.3.zip (145 kB)
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
import nltk
nltk.download('all')

[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/abc.zip.
[nltk_data]    | Downloading package alpino to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/alpino.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping
[nltk_data]    |       taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping
[nltk_data]    |       taggers/averaged_perceptron_tagger_ru.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_rus to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |  

True

In [ ]:
!pip install sacrebleu

In [ ]:
# Import necessary libraries
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.tokenize import word_tokenize
import sacrebleu # For more standardized BLEU scores, and other metrics like CHRF, TER
import pandas as pd # For reading CSV files
import os # For path operations

# --- NLTK Resource Check ---
try:
    nltk.data.find('tokenizers/punkt')
    print("NLTK 'punkt' resource found.")
except (nltk.downloader.DownloadError, LookupError):
    print("NLTK 'punkt' resource not found or incomplete. Attempting to download...")
    try:
        nltk.download('punkt', quiet=False)
        nltk.data.find('tokenizers/punkt')
        print("NLTK 'punkt' resource downloaded and verified successfully.")
    except Exception as e:
        print(f"Failed to download or verify NLTK 'punkt' resource: {e}")
        print("NLTK-based BLEU scores might be affected if 'punkt' is unavailable for English.")

# --- Global File Configuration ---
DATA_FOLDER = "data"
SOURCE_LANGUAGE_CSV_FILENAME = "validation_data.csv"         # English source texts
TARGET_LANGUAGE_CSV_MT1_FILENAME = "validation_data_arabic.csv" # Arabic texts from MT System 1
TARGET_LANGUAGE_CSV_MT2_FILENAME = "output_data_ar.csv"# Arabic texts from MT System 2

def load_data_for_evaluation(data_folder,
                             source_csv_name, source_lang_col,
                             target_ref_csv_name, target_lang_ref_col,
                             target_cand_csv_name, target_lang_cand_col):
    """
    Loads data for evaluating target language translations.
    Allows reference and candidate target texts to come from different CSV files.
    """
    source_language_sentences = []
    target_language_reference_translations = []
    target_language_candidate_translations = []

    source_file_path = os.path.join(data_folder, source_csv_name)
    target_ref_file_path = os.path.join(data_folder, target_ref_csv_name)
    target_cand_file_path = os.path.join(data_folder, target_cand_csv_name)

    # Check existence of all files
    if not os.path.exists(source_file_path):
        print(f"Error: Source language CSV file not found at {source_file_path}")
        return None, None, None
    if not os.path.exists(target_ref_file_path):
        print(f"Error: Target language Reference CSV file not found at {target_ref_file_path}")
        return None, None, None
    if not os.path.exists(target_cand_file_path):
        print(f"Error: Target language Candidate CSV file not found at {target_cand_file_path}")
        return None, None, None

    print(f"Reading Source Language data from: {source_file_path} (Column: {source_lang_col})")
    print(f"Reading Target Language Reference data from: {target_ref_file_path} (Column: {target_lang_ref_col})")
    print(f"Reading Target Language Candidate data from: {target_cand_file_path} (Column: {target_lang_cand_col})")

    try:
        df_source = pd.read_csv(source_file_path)

        # Load target DataFrames. If ref and cand files are the same, load once.
        if target_ref_csv_name == target_cand_csv_name:
            df_target_ref = pd.read_csv(target_ref_file_path)
            df_target_cand = df_target_ref
        else:
            df_target_ref = pd.read_csv(target_ref_file_path)
            df_target_cand = pd.read_csv(target_cand_file_path)

        # Validate required columns
        required_cols_source = {source_lang_col}
        required_cols_target_ref = {target_lang_ref_col}
        required_cols_target_cand = {target_lang_cand_col}

        if not required_cols_source.issubset(df_source.columns):
            print(f"Error: Column '{source_lang_col}' not found in Source CSV: {source_file_path}")
            return None, None, None
        if not required_cols_target_ref.issubset(df_target_ref.columns):
            print(f"Error: Column '{target_lang_ref_col}' (Reference) not found in Target Ref CSV: {target_ref_file_path}")
            return None, None, None
        if not required_cols_target_cand.issubset(df_target_cand.columns):
            print(f"Error: Column '{target_lang_cand_col}' (Candidate) not found in Target Cand CSV: {target_cand_file_path}")
            return None, None, None

        # Align row counts
        min_rows = len(df_source)
        if len(df_target_ref) != min_rows:
            print(f"Warning: Source CSV ({len(df_source)} rows) and Target Reference CSV ({len(df_target_ref)} rows) have different numbers of rows.")
            min_rows = min(min_rows, len(df_target_ref))

        # Check Target Candidate CSV row count against the current min_rows
        # This handles cases where Target Candidate might be different from Target Reference CSV
        if len(df_target_cand) != min_rows :
             # Only print warning if it's actually different from what's already been aligned or from source
            if len(df_target_cand) != len(df_source) or \
               (target_ref_csv_name != target_cand_csv_name and len(df_target_cand) != len(df_target_ref)) or \
               (target_ref_csv_name == target_cand_csv_name and len(df_target_cand) != len(df_target_ref)): # this last condition is redundant if df_target_cand = df_target_ref
                print(f"Warning: Row count mismatch with Target Candidate CSV ({len(df_target_cand)} rows).")
            min_rows = min(min_rows, len(df_target_cand))

        # Apply head(min_rows) if any dataframe was longer
        if len(df_source) > min_rows:
            df_source = df_source.head(min_rows)
            print(f"Adjusted Source CSV to {min_rows} rows.")
        if len(df_target_ref) > min_rows:
            df_target_ref = df_target_ref.head(min_rows)
            print(f"Adjusted Target Reference CSV to {min_rows} rows.")
        if len(df_target_cand) > min_rows: # This check is important if df_target_cand was loaded separately
            df_target_cand = df_target_cand.head(min_rows)
            print(f"Adjusted Target Candidate CSV to {min_rows} rows.")

        if min_rows < len(df_source) or min_rows < len(df_target_ref) or (target_ref_csv_name == target_cand_csv_name and min_rows < len(df_target_cand)) or (target_ref_csv_name != target_cand_csv_name and min_rows < len(df_target_cand)):
             # This condition might be too broad now, but the intent is to inform if alignment happened.
             # A simpler message might be better if min_rows was actually reduced from any initial length.
             if len(df_source.index) != min_rows or len(df_target_ref.index) !=min_rows or len(df_target_cand.index) !=min_rows : # A more direct check
                print(f"Proceeding with {min_rows} aligned rows.")

        for index in range(min_rows):
            source_text = str(df_source.loc[index, source_lang_col])
            ref_target_text = str(df_target_ref.loc[index, target_lang_ref_col])
            cand_target_text = str(df_target_cand.loc[index, target_lang_cand_col])

            if pd.isna(source_text) or pd.isna(ref_target_text) or pd.isna(cand_target_text):
                print(f"Skipping row {index+2} due to missing data in required columns for this configuration.")
                continue

            source_language_sentences.append(source_text)
            target_language_reference_translations.append([ref_target_text])
            target_language_candidate_translations.append(cand_target_text)

    except Exception as e:
        print(f"Error reading or processing CSV files: {e}")
        return None, None, None

    print(f"Loaded {len(source_language_sentences)} sentence pairs for this configuration.")
    return source_language_sentences, target_language_reference_translations, target_language_candidate_translations

def tokenize_target_sentence_nltk(sentence, language='arabic'):
    if language == 'arabic':
        return sentence.split()
    elif language == 'english':
        return word_tokenize(sentence.lower())
    else:
        return sentence.split()

def run_evaluation(config_name,
                   en_src_col,
                   ar_ref_csv, ar_ref_col,
                   ar_cand_csv, ar_cand_col,
                   target_language_code='ar'):
    """
    Runs the full evaluation pipeline for a given configuration.
    """
    print(f"\n--- Running Evaluation for Configuration: {config_name} ---")
    print(f"Source Language (English) Column: '{en_src_col}' (from {SOURCE_LANGUAGE_CSV_FILENAME})")
    print(f"Target Language (Arabic) Reference: File '{ar_ref_csv}', Column '{ar_ref_col}'")
    print(f"Target Language (Arabic) Candidate: File '{ar_cand_csv}', Column '{ar_cand_col}'")

    source_texts, reference_translations, candidate_translations = \
        load_data_for_evaluation(DATA_FOLDER,
                                 SOURCE_LANGUAGE_CSV_FILENAME, en_src_col,
                                 ar_ref_csv, ar_ref_col,
                                 ar_cand_csv, ar_cand_col)

    if not source_texts or not reference_translations or not candidate_translations:
        print(f"Could not load data for configuration '{config_name}'. Skipping.")
        return

    # --- NLTK BLEU ---
    tokenized_references_nltk = []
    tokenized_candidates_nltk = []
    nltk_bleu_calculation_possible = True
    try:
        for refs_list in reference_translations:
            tokenized_refs_for_sentence = [tokenize_target_sentence_nltk(ref, language=target_language_code) for ref in refs_list]
            tokenized_references_nltk.append(tokenized_refs_for_sentence)
        tokenized_candidates_nltk = [tokenize_target_sentence_nltk(cand, language=target_language_code) for cand in candidate_translations]
    except Exception as e:
        print(f"NLTK tokenization error: {e}. NLTK BLEU might be unavailable.")
        nltk_bleu_calculation_possible = False

    average_nltk_bleu = None
    if nltk_bleu_calculation_possible and tokenized_candidates_nltk and tokenized_references_nltk and \
       len(tokenized_candidates_nltk) == len(tokenized_references_nltk):
        nltk_bleu_scores = []
        chencherry = SmoothingFunction()
        for i in range(len(tokenized_candidates_nltk)):
            ref_set = tokenized_references_nltk[i]
            cand_tok = tokenized_candidates_nltk[i]
            if not cand_tok or not any(ref_set): continue
            score = sentence_bleu(ref_set, cand_tok, smoothing_function=chencherry.method1)
            nltk_bleu_scores.append(score)
        if nltk_bleu_scores:
            average_nltk_bleu = sum(nltk_bleu_scores) / len(nltk_bleu_scores)
            print(f"\nAverage NLTK Sentence BLEU (Target: {target_language_code.upper()}): {average_nltk_bleu:.4f}")
        else:
            print("\nCould not calculate NLTK BLEU scores.")
    else:
        print("\nSkipping NLTK BLEU (tokenization issue or data mismatch).")

    # --- SacreBLEU Metrics ---
    corpus_bleu_sacre = None
    corpus_chrf_sacre = None
    corpus_ter_sacre = None
    if len(candidate_translations) == len(reference_translations) and len(candidate_translations) > 0:
        try:
            corpus_bleu_sacre = sacrebleu.corpus_bleu(candidate_translations, reference_translations, tokenize='intl' if target_language_code != 'en' else '13a')
            print(f"SacreBLEU Corpus BLEU (Target: {target_language_code.upper()}): {corpus_bleu_sacre.score:.2f} (BP={corpus_bleu_sacre.bp:.2f} Precisions: {'/'.join([f'{p:.2f}' for p in corpus_bleu_sacre.precisions])})")
        except Exception as e: print(f"Error SacreBLEU BLEU: {e}")
        try:
            corpus_chrf_sacre = sacrebleu.corpus_chrf(candidate_translations, reference_translations)
            print(f"SacreBLEU Corpus CHRF (Target: {target_language_code.upper()}): {corpus_chrf_sacre.score:.2f}")
        except Exception as e: print(f"Error SacreBLEU CHRF: {e}")
        try:
            corpus_ter_sacre = sacrebleu.corpus_ter(candidate_translations, reference_translations)
            print(f"SacreBLEU Corpus TER (Target: {target_language_code.upper()}): {corpus_ter_sacre.score:.2f} (lower is better)")
        except Exception as e: print(f"Error SacreBLEU TER: {e}")
    else:
        print("\nSkipping SacreBLEU (candidate/reference length mismatch or empty).")

    # --- Reporting ---
    print(f"\n--- Summary for Configuration: {config_name} ---")
    print(f"Processed {len(source_texts)} sentence pairs.")
    print(f"Evaluation of Target Language: {target_language_code.upper()}")
    print(f"Average NLTK Sentence BLEU: {average_nltk_bleu:.4f}" if average_nltk_bleu is not None else "Average NLTK Sentence BLEU: Not Available")
    print(f"SacreBLEU Corpus BLEU: {corpus_bleu_sacre.score:.2f}" if corpus_bleu_sacre else "SacreBLEU Corpus BLEU: Not Available")
    print(f"SacreBLEU Corpus CHRF: {corpus_chrf_sacre.score:.2f}" if corpus_chrf_sacre else "SacreBLEU Corpus CHRF: Not Available")
    print(f"SacreBLEU Corpus TER: {corpus_ter_sacre.score:.2f} (lower is better)" if corpus_ter_sacre else "SacreBLEU Corpus TER: Not Available")
    print("--- End of Summary for this Configuration ---")

if __name__ == '__main__':
    print("Starting translation accuracy pipeline (Source: English, Target: Arabic, Multiple MT Systems)...")

    # Define the column types to iterate over
    column_types = ["observation_1", "observation_2", "hypothesis_1", "hypothesis_2"]

    configurations = []

    for col_type in column_types:
        en_source_column = col_type # English source column matches the Arabic column type being evaluated

        # 1. MT1 Self-Evaluation
        configurations.append({
            "name": f"MT1 Self-Eval: Ar_{col_type} (for En_{col_type} src)",
            "en_src_col": en_source_column,
            "ar_ref_csv": TARGET_LANGUAGE_CSV_MT1_FILENAME, "ar_ref_col": col_type,
            "ar_cand_csv": TARGET_LANGUAGE_CSV_MT1_FILENAME, "ar_cand_col": col_type
        })

        # 2. MT2 Self-Evaluation
        configurations.append({
            "name": f"MT2 Self-Eval: Ar_{col_type} (for En_{col_type} src)",
            "en_src_col": en_source_column,
            "ar_ref_csv": TARGET_LANGUAGE_CSV_MT2_FILENAME, "ar_ref_col": col_type,
            "ar_cand_csv": TARGET_LANGUAGE_CSV_MT2_FILENAME, "ar_cand_col": col_type
        })

        # 3. Cross-System: MT2 (candidate) vs MT1 (reference)
        configurations.append({
            "name": f"MT2 vs MT1: Ar_{col_type} (for En_{col_type} src)",
            "en_src_col": en_source_column,
            "ar_ref_csv": TARGET_LANGUAGE_CSV_MT1_FILENAME, "ar_ref_col": col_type,
            "ar_cand_csv": TARGET_LANGUAGE_CSV_MT2_FILENAME, "ar_cand_col": col_type
        })

        # 4. Cross-System: MT1 (candidate) vs MT2 (reference)
        configurations.append({
            "name": f"MT1 vs MT2: Ar_{col_type} (for En_{col_type} src)",
            "en_src_col": en_source_column,
            "ar_ref_csv": TARGET_LANGUAGE_CSV_MT2_FILENAME, "ar_ref_col": col_type,
            "ar_cand_csv": TARGET_LANGUAGE_CSV_MT1_FILENAME, "ar_cand_col": col_type
        })

    # Check file existence before loop
    all_files_exist = True
    # Check all unique CSV files mentioned in configurations
    # For this script, it's always these three, but a more dynamic check could be added if needed.
    required_files = {SOURCE_LANGUAGE_CSV_FILENAME, TARGET_LANGUAGE_CSV_MT1_FILENAME, TARGET_LANGUAGE_CSV_MT2_FILENAME}
    for f_name in required_files:
        if not os.path.exists(os.path.join(DATA_FOLDER, f_name)):
            print(f"Error: Required CSV file not found: {os.path.join(DATA_FOLDER, f_name)}")
            all_files_exist = False

    if all_files_exist:
        for config in configurations:
            run_evaluation(config["name"],
                           config["en_src_col"],
                           config["ar_ref_csv"], config["ar_ref_col"],
                           config["ar_cand_csv"], config["ar_cand_col"],
                           target_language_code='ar')
        print("\nAll configured evaluations complete.")
    else:
        print("\nPipeline halted due to missing CSV files. Please check the 'data' folder and filenames.")



NLTK 'punkt' resource found.
Starting translation accuracy pipeline (Source: English, Target: Arabic, Multiple MT Systems)...

--- Running Evaluation for Configuration: MT1 Self-Eval: Ar_observation_1 (for En_observation_1 src) ---
Source Language (English) Column: 'observation_1' (from validation_data.csv)
Target Language (Arabic) Reference: File 'validation_data_arabic.csv', Column 'observation_1'
Target Language (Arabic) Candidate: File 'validation_data_arabic.csv', Column 'observation_1'
Reading Source Language data from: data/validation_data.csv (Column: observation_1)
Reading Target Language Reference data from: data/validation_data_arabic.csv (Column: observation_1)
Reading Target Language Candidate data from: data/validation_data_arabic.csv (Column: observation_1)
Loaded 1532 sentence pairs for this configuration.

Average NLTK Sentence BLEU (Target: AR): 0.9820
SacreBLEU Corpus BLEU (Target: AR): 100.00 (BP=1.00 Precisions: 100.00/100.00/100.00/100.00)
SacreBLEU Corpus CHRF (T